# Ch19: Evaluation (评估) — LangChain + MiMo

本章演示如何评估 Agent 系统的输出质量，包含两种方法：

1. **基础评估** — 准确率、延迟、Token 使用量（纯 Python）
2. **LLM-as-Judge** — 用 LLM 作为评判者，多维度打分和反馈

## 为什么需要评估？
- 没有评估就没有改进——评估是 Agent 系统持续优化的基础
- 基础评估告诉你「快不快、省不省」，LLM-as-Judge 告诉你「好不好」
- 生产环境必须有评估闭环：部署 → 收集数据 → 评估 → 优化 → 重新部署

## 第1步：导入依赖

In [1]:
import os
import time
import json
from typing import Optional

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

os.environ["OPENAI_API_KEY"] = os.getenv("MIMO_API_KEY", "")
os.environ["OPENAI_API_BASE"] = "https://token-plan-cn.xiaomimimo.com/v1"

llm = ChatOpenAI(
    model="mimo-v2.5-pro",
    temperature=0.2,
    base_url="https://token-plan-cn.xiaomimimo.com/v1",
)
print("MiMo 模型初始化完成")

MiMo 模型初始化完成


## 第2步：基础评估（纯 Python）

三种基础评估指标，不依赖 LLM：
- **准确率**：精确匹配 agent 输出与期望输出
- **延迟**：测量函数执行时间
- **Token 监控**：统计输入/输出 token 数

In [2]:
# --- 1. 响应准确率评估 ---

def evaluate_response_accuracy(agent_output: str, expected_output: str) -> float:
    """精确匹配评估：完全一致得 1.0，否则得 0.0"""
    return 1.0 if agent_output.strip().lower() == expected_output.strip().lower() else 0.0

# 测试
agent_response = "法国的首都是巴黎"
ground_truth = "法国的首都是巴黎"
score = evaluate_response_accuracy(agent_response, ground_truth)
print(f"精确匹配得分: {score}")

# 模糊匹配版本（包含即可）
def evaluate_response_fuzzy(agent_output: str, expected_output: str) -> float:
    """模糊匹配：期望输出出现在 agent 输出中即为正确"""
    return 1.0 if expected_output.strip() in agent_output.strip() else 0.0

agent_response2 = "根据资料，法国的首都是巴黎。巴黎位于塞纳河畔。"
score2 = evaluate_response_fuzzy(agent_response2, "巴黎")
print(f"模糊匹配得分: {score2}")

精确匹配得分: 1.0
模糊匹配得分: 1.0


In [3]:
# --- 2. 延迟测量 ---

def timed_agent_action(agent_function, *args, **kwargs):
    """测量 agent 函数的执行时间"""
    start_time = time.perf_counter()
    result = agent_function(*args, **kwargs)
    end_time = time.perf_counter()
    latency_ms = (end_time - start_time) * 1000
    print(f"函数 '{agent_function.__name__}' 耗时: {latency_ms:.2f} ms")
    return result, latency_ms

# 测试：模拟一个工具调用
def simulated_tool_call(query):
    time.sleep(0.15)  # 模拟耗时
    return f"查询结果: {query}"

result, latency = timed_agent_action(simulated_tool_call, "查询天气")
print(f"结果: {result}")

函数 'simulated_tool_call' 耗时: 150.37 ms


结果: 查询结果: 查询天气


In [4]:
# --- 3. Token 使用监控 ---

class LLMInteractionMonitor:
    """LLM 交互 token 监控器"""
    def __init__(self):
        self.total_input_tokens = 0
        self.total_output_tokens = 0
        self.interactions = []

    def record_interaction(self, prompt: str, response: str):
        """记录一次交互（用 split 做粗略 token 估算）"""
        input_tokens = len(prompt.split())
        output_tokens = len(response.split())
        self.total_input_tokens += input_tokens
        self.total_output_tokens += output_tokens
        self.interactions.append({
            "input_tokens": input_tokens,
            "output_tokens": output_tokens
        })
        print(f"记录交互: 输入 tokens={input_tokens}, 输出 tokens={output_tokens}")

    def get_summary(self):
        """获取汇总统计"""
        return {
            "总交互次数": len(self.interactions),
            "总输入 tokens": self.total_input_tokens,
            "总输出 tokens": self.total_output_tokens,
            "总 tokens": self.total_input_tokens + self.total_output_tokens
        }

# 测试
monitor = LLMInteractionMonitor()
monitor.record_interaction("法国的首都是哪里？", "法国的首都是巴黎。")
monitor.record_interaction("讲个笑话。", "为什么科学家不信任原子？因为它们组成了一切！")
summary = monitor.get_summary()
print(f"\n汇总统计: {json.dumps(summary, ensure_ascii=False, indent=2)}")

记录交互: 输入 tokens=1, 输出 tokens=1
记录交互: 输入 tokens=1, 输出 tokens=1

汇总统计: {
  "总交互次数": 2,
  "总输入 tokens": 2,
  "总输出 tokens": 2,
  "总 tokens": 4
}


## 第3步：LLM-as-Judge（用 LLM 作为评判者）

用 LLM 评估输出质量，定义结构化评分标准：
- **清晰度**（1-5分）：表述是否明确、无歧义
- **中立性**（1-5分）：是否存在偏见或引导性
- **相关性**（1-5分）：是否紧扣主题
- **完整性**（1-5分）：信息是否充分
- **总分** + 详细反馈 + 改进建议

In [5]:
# LLM-as-Judge 评分标准

JUDGE_RUBRIC = """
你是一位专业的质量评估专家。请评估以下回答的质量。

请从以下维度打分（每项 1-5 分）：

1. **清晰度**：
   - 1分：极其模糊、令人困惑
   - 3分：基本清晰，但可以更精确
   - 5分：完全清晰、无歧义

2. **中立性**：
   - 1分：严重偏向某一方，有明显引导
   - 3分：略有倾向，但总体客观
   - 5分：完全中立、客观

3. **相关性**：
   - 1分：与问题完全无关
   - 3分：部分相关，但有偏离
   - 5分：完全切题，重点突出

4. **完整性**：
   - 1分：严重缺失关键信息
   - 3分：基本完整，细节略有不足
   - 5分：信息充分、全面

请严格以 JSON 格式输出：
{
  "overall_score": 整数(1-5),
  "scores": {"clarity": 分数, "neutrality": 分数, "relevance": 分数, "completeness": 分数},
  "rationale": "评分理由",
  "feedback": ["改进建议1", "改进建议2"],
  "recommended_action": "建议操作"
}
"""

def llm_as_judge(question: str, answer: str) -> Optional[dict]:
    """用 LLM 作为评判者评估回答质量"""
    prompt = JUDGE_RUBRIC + f"\n\n---\n**问题：**\n{question}\n\n**待评估的回答：**\n{answer}\n---"

    try:
        response = llm.invoke([HumanMessage(content=prompt)])
        text = response.content.strip()
        # 如果包含 markdown 代码块标记，提取其中的 JSON
        if "```json" in text:
            text = text.split("```json")[1].split("```")[0].strip()
        elif "```" in text:
            text = text.split("```")[1].split("```")[0].strip()
        return json.loads(text)
    except json.JSONDecodeError as e:
        print(f"JSON 解析失败: {e}")
        print(f"原始输出: {response.content}")
        return None
    except Exception as e:
        print(f"评估出错: {e}")
        return None

print("LLM-as-Judge 函数定义完成")

LLM-as-Judge 函数定义完成


## 第4步：测试 LLM-as-Judge

用三个不同质量的示例来测试：
1. 高质量回答（准确、完整）
2. 有偏见的回答（引导性语言）
3. 模糊的回答（信息不足）

In [6]:
# 测试案例 1：高质量回答
print("=" * 60)
print("测试 1：高质量回答")
print("=" * 60)

question1 = "什么是 RAG（检索增强生成）？"
answer1 = "RAG（Retrieval-Augmented Generation）是一种将信息检索与大语言模型结合的技术。它的工作流程是：1）接收用户问题；2）从知识库中检索相关文档；3）将检索结果注入提示词；4）LLM 基于检索结果生成答案。RAG 的优势在于可以减少幻觉、支持私有数据、无需重新训练模型。"

result1 = llm_as_judge(question1, answer1)
if result1:
    print(json.dumps(result1, ensure_ascii=False, indent=2))

测试 1：高质量回答


{
  "overall_score": 5,
  "scores": {
    "clarity": 5,
    "neutrality": 5,
    "relevance": 5,
    "completeness": 4
  },
  "rationale": "该回答质量很高。它准确、清晰地定义了RAG，并通过结构化的步骤（1-4）清晰地阐述了其工作流程，最后总结了核心优势。回答完全中立，仅陈述技术事实，无任何倾向性。内容与问题高度相关，直接、全面地回答了“什么是RAG”这一核心问题。完整性方面，它涵盖了RAG的核心概念、流程和主要优势，足以让提问者建立基本理解，但若能简要提及实现该技术所需的关键组件（如向量数据库、嵌入模型）或其局限性，会更为全面。",
  "feedback": [
    "可以补充一两个关键的技术实现细节，例如‘通常需要将文档转换为向量并存储在向量数据库中以便高效检索’，这能让技术画像更立体。",
    "在优势部分，可以简要提及一个潜在的挑战或局限性，例如‘其效果高度依赖于检索到的文档质量’，以使描述更平衡。"
  ],
  "recommended_action": "采纳该回答。它准确、清晰且完整地解答了用户的问题，质量优秀。"
}


In [7]:
# 测试案例 2：有偏见的回答
print("=" * 60)
print("测试 2：有偏见的回答")
print("=" * 60)

question2 = "RAG 和微调哪个更好？"
answer2 = "RAG 明显优于微调。微调成本高、容易过拟合，而且每次更新数据都要重新训练。RAG 简单高效，任何场景都应该优先选择 RAG。"

result2 = llm_as_judge(question2, answer2)
if result2:
    print(json.dumps(result2, ensure_ascii=False, indent=2))

测试 2：有偏见的回答


{
  "overall_score": 2,
  "scores": {
    "clarity": 5,
    "neutrality": 1,
    "relevance": 5,
    "completeness": 1
  },
  "rationale": "该回答在清晰度和相关性上表现优秀，观点表达直接且紧扣问题。然而，其核心缺陷在于严重缺乏中立性和完整性。回答完全偏向RAG，使用了‘明显优于’、‘任何场景都应该优先选择’等绝对化、引导性语言，未能客观呈现微调在特定场景下的优势（如深度定制、数据隐私、离线环境等）。信息呈现片面，仅列举了微调的缺点和RAG的优点，导致回答不完整、不客观，整体质量较低。",
  "feedback": [
    "保持中立客观：在比较两种技术时，应避免使用‘明显优于’、‘任何场景’等绝对化表述。应客观陈述各自的优势、劣势和适用场景。",
    "补充关键信息：回答应补充微调的优势，例如：在需要深度领域知识定制、模型行为高度可控、数据隐私要求极高或网络环境受限的场景下，微调可能是更优或唯一的选择。同时，也应提及RAG的局限性，如依赖外部知识库的质量、可能引入无关信息、增加系统复杂性和延迟等。",
    "提供决策框架：更高质量的回答应引导用户根据具体需求（如成本、数据更新频率、定制化程度、部署环境）来选择，而非给出单一结论。"
  ],
  "recommended_action": "重写回答，提供一个平衡、全面的比较。结构可以包括：1) 简述两者核心原理；2) 分别列出RAG和微调的主要优势与劣势；3) 总结各自的典型适用场景；4) 给出选择建议的考量因素。"
}


In [8]:
# 测试案例 3：模糊的回答
print("=" * 60)
print("测试 3：模糊的回答")
print("=" * 60)

question3 = "什么是 Agent？"
answer3 = "Agent 就是智能体，能做很多事情。"

result3 = llm_as_judge(question3, answer3)
if result3:
    print(json.dumps(result3, ensure_ascii=False, indent=2))

测试 3：模糊的回答


{
  "overall_score": 1,
  "scores": {
    "clarity": 1,
    "neutrality": 5,
    "relevance": 3,
    "completeness": 1
  },
  "rationale": "该回答过于简略，未能提供任何实质性信息。它仅给出了一个同义词替换（“智能体”）和一个极其模糊的描述（“能做很多事情”），完全没有解释Agent的核心概念、特征、工作原理或应用场景。这导致回答在清晰度和完整性上严重不足，无法满足用户获取知识的基本需求。虽然回答本身没有表现出偏见（中立性高），且与问题主题相关（相关性中等），但其信息量几乎为零，因此整体质量极低。",
  "feedback": [
    "提供清晰的定义：应解释Agent（智能体）通常指能够感知环境、做出决策并采取行动以实现特定目标的自主实体或系统。",
    "补充关键信息：需要说明Agent的核心特征（如自主性、反应性、主动性、社交能力等），并举例说明其应用（如软件Agent、游戏中的NPC、自动驾驶系统等）。",
    "结构化回答：建议采用“定义 + 核心特征 + 应用示例”的结构，使回答更完整、易于理解。"
  ],
  "recommended_action": "要求回答者重新生成一个更详细、结构化的回答，以全面解释Agent的概念。"
}


## 第5步：批量评估 + 结果汇总

将多个评估结果汇总，生成评估报告。

In [9]:
# 批量评估并汇总

test_cases = [
    {
        "question": "什么是 RAG？",
        "answer": "RAG（Retrieval-Augmented Generation）是一种将信息检索与大语言模型结合的技术。它的工作流程是：1）接收用户问题；2）从知识库中检索相关文档；3）将检索结果注入提示词；4）LLM 基于检索结果生成答案。RAG 的优势在于可以减少幻觉、支持私有数据、无需重新训练模型。",
        "expected_quality": "高"
    },
    {
        "question": "RAG 和微调哪个更好？",
        "answer": "RAG 明显优于微调。微调成本高、容易过拟合，任何场景都应该优先选择 RAG。",
        "expected_quality": "中（有偏见）"
    },
    {
        "question": "什么是 Agent？",
        "answer": "Agent 就是智能体，能做很多事情。",
        "expected_quality": "低（模糊）"
    },
]

results = []
for i, case in enumerate(test_cases):
    print(f"\n评估第 {i+1}/{len(test_cases)} 个回答...")
    judgment = llm_as_judge(case["question"], case["answer"])
    if judgment:
        results.append({
            "问题": case["question"],
            "期望质量": case["expected_quality"],
            "总分": judgment.get("overall_score", "N/A"),
            "各维度分数": judgment.get("scores", {}),
            "评分理由": judgment.get("rationale", ""),
            "建议": judgment.get("recommended_action", "")
        })

# 汇总报告
print("\n" + "=" * 60)
print("评估汇总报告")
print("=" * 60)
for i, r in enumerate(results):
    print(f"\n--- 回答 {i+1} ---")
    print(f"问题: {r['问题']}")
    print(f"期望质量: {r['期望质量']}")
    print(f"总分: {r['总分']}/5")
    if isinstance(r['各维度分数'], dict):
        for dim, score in r['各维度分数'].items():
            print(f"  {dim}: {score}/5")
    print(f"理由: {r['评分理由']}")
    print(f"建议: {r['建议']}")


评估第 1/3 个回答...



评估第 2/3 个回答...



评估第 3/3 个回答...



评估汇总报告

--- 回答 1 ---
问题: 什么是 RAG？
期望质量: 高
总分: 5/5
  clarity: 5/5
  neutrality: 5/5
  relevance: 5/5
  completeness: 5/5
理由: 该回答在所有评估维度上均表现出色。它准确、清晰地定义了RAG，用四个步骤清晰地描述了其工作流程，并列举了其核心优势。回答完全中立，仅陈述事实，没有倾向性。内容与问题高度相关，聚焦于解释RAG是什么。对于“什么是RAG”这一基础问题，回答提供了定义、流程和优势三个关键层面的信息，信息充分且全面，足以让提问者建立清晰的概念。
建议: 直接采纳该回答。

--- 回答 2 ---
问题: RAG 和微调哪个更好？
期望质量: 中（有偏见）
总分: 2/5
  clarity: 5/5
  neutrality: 1/5
  relevance: 3/5
  completeness: 1/5
理由: 回答观点清晰（清晰度5分），且与问题直接相关（相关性3分）。然而，其核心缺陷在于严重缺乏中立性（1分）和完整性（1分）。回答以绝对化、片面的口吻做出了结论（“明显优于”、“任何场景都应该”），完全忽略了两种技术各自的优缺点、适用场景以及相互补充的可能性，未能提供任何支撑论据或背景信息，属于严重缺失关键信息。
建议: 重新生成一个更平衡、更全面的回答，客观比较RAG和微调在不同维度（如成本、数据需求、知识更新、任务特异性、延迟等）上的表现，并指出它们并非互斥，常可结合使用。

--- 回答 3 ---
问题: 什么是 Agent？
期望质量: 低（模糊）
总分: 1/5
  clarity: 1/5
  neutrality: 5/5
  relevance: 3/5
  completeness: 1/5
理由: 该回答过于简略，仅提供了术语对应（Agent=智能体）和一句极其模糊的描述（能做很多事情）。它没有解释Agent的核心概念、主要特征、工作原理或应用场景，导致信息量严重不足，无法让提问者理解“Agent”的实质。虽然回答是中立的，且与问题主题相关，但因其极度不清晰和不完整，整体质量很低。
建议: 重新生成一个更详细、更专业的回答，确保涵盖Agent的核心定义、关键特征、基本分类和典型应用。


## 总结

### 评估方法对比

| 方法 | 优点 | 缺点 | 适用场景 |
|------|------|------|----------|
| 精确匹配 | 简单、无成本 | 无法处理近似正确 | 事实性问答 |
| 模糊匹配 | 容忍格式差异 | 仍不够智能 | 关键词提取 |
| 延迟测量 | 精确量化性能 | 不评估质量 | 性能优化 |
| Token 监控 | 控制成本 | 不评估质量 | 成本管理 |
| LLM-as-Judge | 多维度、有反馈 | 有成本、可能不一致 | 开放式回答评估 |

### 核心要点
- **基础评估**（准确率/延迟/Token）是底线监控，成本低，必须有
- **LLM-as-Judge** 是高级评估，适合开放式回答，但需注意评估一致性
- 生产环境建议：基础评估实时监控 + LLM-as-Judge 定期抽检
- 评估标准要根据业务场景定制，不要照搬通用模板